In [ ]:
import numpy as np
import torch
import torch.nn as nn
# same seeds as training. 
torch.manual_seed(0)
np.random.seed(0)

In [ ]:
class GridLockedCNN(nn.Module):
    def __init__(self, hidden=64, kernel_size=5, num_layers=4):
        super().__init__()
        layers = [nn.Conv1d(1, hidden, kernel_size,
                            padding=kernel_size // 2,
                            padding_mode='circular'),
                  nn.ReLU()]
        for _ in range(num_layers - 1):
            layers += [nn.Conv1d(hidden, hidden, kernel_size,
                                 padding=kernel_size // 2,
                                 padding_mode='circular'),
                       nn.ReLU()]
        layers.append(nn.Conv1d(hidden, 1, kernel_size,
                                padding=kernel_size // 2,
                                padding_mode='circular'))
        self.net = nn.Sequential(*layers)

    def forward(self, u):
        return u + self.net(u)

In [ ]:
def rel_l2_per_sample(pred, target):
    num = np.linalg.norm((pred - target).reshape(len(pred), -1), axis=1)
    den = np.linalg.norm(target.reshape(len(target), -1), axis=1) + 1e-8
    return num / den

In [ ]:
device = 'cpu'
model = GridLockedCNN(hidden=64, kernel_size=3, num_layers=4).to(device)
model.load_state_dict(torch.load('cnn_128.pt', map_location=device))
model.eval()
print("Loaded cnn_128.pt\n")

In [ ]:
# apply the same trained model to all three resolutions, no retraining
results = {}
with torch.no_grad():
    for res in [128, 256, 512]:
        data = np.load(f'eval_{res}.npz')
        X = data['field'][:, None, :].astype(np.float32)
        Y = data['target'][:, None, :].astype(np.float32)

        pred = model(torch.tensor(X)).numpy()

        cnn_err = rel_l2_per_sample(pred, Y)
        id_err = rel_l2_per_sample(X, Y)

        results[res] = {'cnn': cnn_err, 'identity': id_err}

        print(f"=== Resolution {res} ===")
        print(f"  CNN mean rel-L2   : {cnn_err.mean():.6f}")
        print(f"  CNN median        : {np.median(cnn_err):.6f}")
        print(f"  Identity baseline : {id_err.mean():.6f}")
        print(f"  Improvement factor: {id_err.mean() / cnn_err.mean():.2f}x")
        print()

In [ ]:
e128 = results[128]['cnn'].mean()
e256 = results[256]['cnn'].mean()
e512 = results[512]['cnn'].mean()

Growth ratio = error at higher resolution / error at training resolution.
Ratio ~1 means resolution-invariant; ratio is 1 means the model degrades, which is exactly what we expect from a grid-locked CNN.

In [ ]:
print("=== Error Growth ===")
print(f"  Growth 128 -> 256: {e256/e128:.2f}x")
print(f"  Growth 128 -> 512: {e512/e128:.2f}x")

In [ ]:
# per-example errors for the buckets 
np.savez('cnn_errors.npz',
         err_128=results[128]['cnn'],
         err_256=results[256]['cnn'],
         err_512=results[512]['cnn'],
         id_128=results[128]['identity'],
         id_256=results[256]['identity'],
         id_512=results[512]['identity'])
print("\nSaved per-example errors to cnn_errors.npz")